# 4 · Relationships Between Variables
*Data Visualization for Scientists & Public Health Professionals*

When you have two numeric variables and want to know how they move together, the **scatter plot** is the basic tool, and the **correlation heatmap** is how you scan many pairs at once. This notebook covers scatter plots, the overplotting that shows up at real sample sizes, adding a third variable through color or size, fitting a trend line, reading a heatmap, and — as a first taste of interactivity — a hover-enabled scatter for exploring individual points.

We join two county-level tables, ACS demographics and CDC PLACES health prevalences, and ask a real question: does county poverty track diabetes prevalence?

### Learning objectives
- Draw a scatter plot of two numeric variables
- Recognize and fix overplotting
- Encode a third variable with hue or marker size
- Add a trend line, and state its caveats
- Build and read a correlation heatmap
- Make a hover-to-explore scatter when the reader needs individual points

### Agenda
1. The scatter plot
2. Overplotting
3. A third variable: hue and size
4. Trend lines
5. The correlation heatmap
6. Hover to explore

### How the exercises work
Each exercise has a prompt, an empty cell to try it yourself, and a collapsed **Solution** you can expand to check your work.

## Setup

We join two county-level tables on the county FIPS code — `CountyId` in ACS, `CountyFIPS` in PLACES, both integers. You covered joins in the pandas course, so here we just build the combined frame and move on to the plotting. The `h`-prefixed columns are PLACES modeled prevalence estimates — the percent of adults in the county with each condition (`hdiabetes`, `hbphigh` high blood pressure, `hchd` coronary heart disease, and so on).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

BASE_URL = "https://raw.githubusercontent.com/jimcody2014/2026-python-data/refs/heads/main"
acs = pd.read_csv(f"{BASE_URL}/acs2017.csv")
places = pd.read_csv(f"{BASE_URL}/places.csv")

counties = acs.merge(places, left_on="CountyId", right_on="CountyFIPS", how="inner")
print(counties.shape)
counties[["County", "State", "Poverty", "Income", "hdiabetes", "hbphigh"]].head()

## 1. The scatter plot

One point per county: poverty rate on x, diabetes prevalence on y. The eye reads the tilt of the cloud as the relationship. `sns.scatterplot` is the tool.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
sns.scatterplot(data=counties, x="Poverty", y="hdiabetes", ax=ax)
ax.set_title("County poverty vs. diabetes prevalence")
ax.set_xlabel("Poverty rate (%)")
ax.set_ylabel("Diabetes prevalence (%)")
fig.tight_layout()
plt.show()

> **Note:** The cloud tilts up to the right: counties with more poverty tend to have more diabetes (a correlation around 0.7). A scatter shows *association*, not cause — a caveat worth saying out loud to any audience. [seaborn relational tutorial](https://seaborn.pydata.org/tutorial/relational.html)

## 2. Overplotting

With about 3,100 counties, points pile on top of one another and you can no longer see where the data are densest — a solid blob hides its own shape. The first fix is **transparency** (`alpha`) plus a smaller marker, so dense regions read darker.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)

sns.scatterplot(data=counties, x="Poverty", y="hdiabetes", ax=axes[0])
axes[0].set_title("Opaque: the dense core is a blob")
axes[0].set_xlabel("Poverty rate (%)")
axes[0].set_ylabel("Diabetes prevalence (%)")

sns.scatterplot(data=counties, x="Poverty", y="hdiabetes", alpha=0.25, s=18, ax=axes[1])
axes[1].set_title("alpha=0.25: density becomes visible")
axes[1].set_xlabel("Poverty rate (%)")

fig.tight_layout()
plt.show()

> **Tip:** When even transparency saturates, switch to a 2-D histogram (`sns.histplot` with both `x` and `y`) or `sns.jointplot(kind="hex")`, which bin the points and color by count instead of drawing each one.

### Feel the overplotting fix

Run the cell and drag the two sliders — transparency and marker size. Watch the shapeless blob resolve into a cloud with a visible dense core, and notice there is a sweet spot: too transparent or too small and the sparse edges vanish. (The sliders need a live kernel.)

In [ ]:
from ipywidgets import interact, FloatSlider, IntSlider

def scatter(alpha=0.25, size=18):
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.scatterplot(data=counties, x="Poverty", y="hdiabetes", alpha=alpha, s=size, ax=ax)
    ax.set_title(f"alpha={alpha:.2f}, size={size}")
    ax.set_xlabel("Poverty rate (%)")
    ax.set_ylabel("Diabetes prevalence (%)")
    fig.tight_layout()
    plt.show()

interact(scatter,
         alpha=FloatSlider(min=0.05, max=1.0, step=0.05, value=0.25),
         size=IntSlider(min=5, max=60, step=5, value=18));

## 3. A third variable: hue and size

A scatter can carry more than two variables. **`hue=`** colors points by a third (a gradient for a continuous variable, distinct colors for a categorical one with a few levels). **`size=`** scales the marker by a fourth — a bubble chart. Use them sparingly; each extra encoding asks more of the reader.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

# color by income (continuous -> gradient)
sns.scatterplot(data=counties, x="Poverty", y="hdiabetes",
                hue="Income", alpha=0.6, s=18, ax=axes[0])
axes[0].set_title("Color = median income")
axes[0].set_xlabel("Poverty rate (%)")
axes[0].set_ylabel("Diabetes prevalence (%)")

# size by population (continuous -> bubble)
sns.scatterplot(data=counties, x="Poverty", y="hdiabetes",
                size="TotalPop", sizes=(5, 200), alpha=0.4, legend=False, ax=axes[1])
axes[1].set_title("Size = county population")
axes[1].set_xlabel("Poverty rate (%)")
axes[1].set_ylabel("Diabetes prevalence (%)")

fig.tight_layout()
plt.show()

> **Note:** The income gradient reinforces the story — the high-poverty, high-diabetes corner is also where income is lowest. For a *categorical* third variable, pass it to `hue=` the same way; keep it to a handful of levels or the colors stop being distinguishable.

## 4. Trend lines

To summarize the relationship with a fitted line, `sns.regplot` draws the scatter plus a linear fit and a shaded confidence band in one call.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
sns.regplot(data=counties, x="Poverty", y="hdiabetes",
            scatter_kws={"alpha": 0.2, "s": 15}, line_kws={"color": "crimson"}, ax=ax)
ax.set_title("Poverty vs. diabetes, with linear fit")
ax.set_xlabel("Poverty rate (%)")
ax.set_ylabel("Diabetes prevalence (%)")
fig.tight_layout()
plt.show()

> **Tip:** A straight line assumes the relationship *is* straight. When you suspect a curve, `lowess=True` fits a flexible local smooth instead (that option needs the `statsmodels` package). To fit separate lines per group, `sns.lmplot(...)` takes a `hue=` and draws one line each. [seaborn regplot](https://seaborn.pydata.org/generated/seaborn.regplot.html)

### Exercise 1 — Straight line or curve? *(7 min)*

Fit the **`Income`** vs **`hdiabetes`** relationship two ways, side by side: a straight-line `regplot` on the left, and the same with `lowess=True` on the right (this needs the `statsmodels` package). In a comment, decide whether the straight line is an honest summary here, or whether it hides a curve.

In [ ]:
# Your work here


<details>
<summary><b>Solution</b></summary>

```python
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5), sharey=True)
sns.regplot(data=counties, x="Income", y="hdiabetes",
            scatter_kws={"alpha": 0.2, "s": 15},
            line_kws={"color": "crimson"}, ax=axes[0])
axes[0].set_title("Straight-line fit")
sns.regplot(data=counties, x="Income", y="hdiabetes", lowess=True,
            scatter_kws={"alpha": 0.2, "s": 15},
            line_kws={"color": "crimson"}, ax=axes[1])
axes[1].set_title("LOWESS (flexible) fit")
for ax in axes:
    ax.set_xlabel("Median household income ($)")
axes[0].set_ylabel("Diabetes prevalence (%)")
fig.tight_layout()
plt.show()
# Does the straight line miss a curve? Diabetes falls steeply as income rises out of
# poverty, then flattens -- the LOWESS bends where the line cannot. Is the linear summary
# honest here, or hiding curvature?
```

**Why this works.** Both panels fit the same two columns; only the model differs. `regplot` draws a straight ordinary-least-squares line by default, while `lowess=True` fits a flexible local smooth (via `statsmodels`) that can bend. Comparing them makes the honesty question concrete: where the smooth curves but the line stays straight, the linear summary is hiding structure.

</details>

## 5. The correlation heatmap

A scatter shows one pair at a time. To scan **many** pairs at once, compute a correlation matrix and draw it as a heatmap: each cell is the correlation between a row variable and a column variable, colored by value. Use a **diverging** colormap centered at zero so the sign reads as color, and fix the scale to the full −1 to 1 range so the colors are comparable.

In [ ]:
cols = ["Poverty", "Income", "Unemployment",
        "hdiabetes", "hbphigh", "hcopd", "hchd", "hstroke"]
corr = counties[cols].corr()

fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag",
            center=0, vmin=-1, vmax=1, square=True, ax=ax)
ax.set_title("Correlations: county demographics and health")
fig.tight_layout()
plt.show()

> **Note:** Read it by scanning for strong color. The diagonal is always 1 (each variable with itself). `Poverty` and `Income` sit at a deep negative (about −0.75); the chronic conditions cluster in warm positives with one another and with poverty. `center=0` and the fixed `vmin`/`vmax` are what keep the colors honest. [seaborn heatmap](https://seaborn.pydata.org/generated/seaborn.heatmap.html)

### Exercise 2 — A masked heatmap *(8 min)*

Build a correlation heatmap of the health measures only — `hdiabetes`, `hbphigh`, `hhighchol`, `hchd`, `hstroke`, `hcopd` — but **mask the upper triangle** with `np.triu` so each pair appears once. Diverging colormap, centered at zero. In a comment, explain why a full square wastes half its ink, and name the strongest pair.

In [ ]:
# Your work here


<details>
<summary><b>Solution</b></summary>

```python
health = ["hdiabetes", "hbphigh", "hhighchol", "hchd", "hstroke", "hcopd"]
corr = counties[health].corr()
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)   # hide the duplicate upper half

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="vlag",
            center=0, vmin=-1, vmax=1, square=True, ax=ax)
ax.set_title("Chronic-condition correlations (lower triangle only)")
fig.tight_layout()
plt.show()
# A correlation matrix is symmetric, so the upper triangle just repeats the lower. Masking
# it drops the duplicate numbers and makes the strong pairs easier to find. Which pair is
# strongest now? (diabetes & stroke, ~0.92)
```

**Why this works.** A correlation matrix is symmetric, so the upper triangle merely mirrors the lower — half the numbers are duplicates. `np.triu(..., k=1)` builds a boolean mask of that redundant half, and `mask=` hides it, leaving each pair once. The result is far easier to scan for the strongest relationships, and it is a genuine heatmap habit rather than a redraw on fewer columns.

</details>

## 6. Hover to explore

Every scatter so far has a limit: you can see the *shape*, but not *which county* any point is. When a reader needs to interrogate individual points, an interactive chart earns its place. `plotly` — a different library, used here purely for this — draws a scatter you can hover, zoom, and pan; hovering a point reveals the county and its numbers.

In [ ]:
fig = px.scatter(counties, x="Poverty", y="hdiabetes",
                 hover_name="County", hover_data=["State", "Income"],
                 opacity=0.5,
                 labels={"Poverty": "Poverty rate (%)", "hdiabetes": "Diabetes prevalence (%)"},
                 title="County poverty vs. diabetes (hover to explore)")
fig.show()

> **Tip:** This renders as an interactive chart in Colab or Jupyter; a statically viewed notebook shows nothing, because the interactivity *is* the point — hover to name the outlier county, zoom into the dense core, pan around. Reach for this when exploration matters; for a finished static figure in a paper or slide, the matplotlib/seaborn tools are still the right choice. (Building full interactive dashboards is a separate course.)

## Wrap-up

You can now draw a scatter of two numeric variables, defeat overplotting with transparency or binning, layer in a third variable with hue or size, fit and caveat a trend line, read a whole correlation matrix as a heatmap with an honest diverging scale, and hand the reader a hover-to-explore scatter when individual points matter.

**Next:** Time series — plotting over dates, resampling and rolling smoothing, handling gaps honestly, and annotating the events that explain the data.